# Training SFT (QLoRA) — CoT vs non-CoT
Latih student `Qwen2.5-1.5B`/`0.5B` dari `data/sft/cot.jsonl` & `nocot.jsonl`.
Hyperparam identik, beda **cuma dataset** -> isolasi efek CoT.

**Setting Kaggle:** GPU **T4 x2** (1 GPU cukup utk 1.5B QLoRA), Internet **On**.
Add Input = dataset berisi `cot.jsonl` + `nocot.jsonl` (output `to_chatml.py`).

### 1. Install

In [ ]:
!pip install -q -U unsloth unsloth_zoo
!pip install -q -U datasets pyyaml

### 2. Clone/pull repo (permanen)
Repo private -> butuh PAT. Simpan Kaggle Secret `GH_TOKEN` (fine-grained, Contents:Read).
Kalau repo sudah public, hapus bagian token.

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = '/kaggle/working/FP_NLP'
TOKEN = ''
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret('GH_TOKEN')
except Exception as e:
    print('no GH_TOKEN secret (ok kalau repo public):', e)
auth = f'{TOKEN}@' if TOKEN else ''
URL = f'https://{auth}github.com/henray404/FP_NLP.git'
if os.path.exists(REPO):
    subprocess.run(['git','-C',REPO,'pull','-q'], check=True)
else:
    subprocess.run(['git','clone','-q',URL,REPO], check=True)
sys.path.insert(0, REPO)
os.chdir(REPO)
print('repo ready:', REPO)

### 3. Temukan dataset SFT
Cari `cot.jsonl`/`nocot.jsonl` di `/kaggle/input`, salin ke `data/sft/`.

In [ ]:
import glob, shutil
Path('data/sft').mkdir(parents=True, exist_ok=True)
for name in ['cot.jsonl','nocot.jsonl']:
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if hits:
        shutil.copy(hits[0], f'data/sft/{name}')
        print(name, '<-', hits[0])
    else:
        print('!! tidak ketemu', name)

### 4. Konfigurasi run
Pilih config (1.5B default). `MAX_EXAMPLES` kecil dulu buat smoke test, lalu `None` utk full.

In [ ]:
from src.training.train_sft import load_config, build_and_train
CONFIGS = ['src/training/configs/cot_1.5b.yaml',
           'src/training/configs/nocot_1.5b.yaml']
MAX_EXAMPLES = 50   # None = full
OUT_ROOT = '/kaggle/working'

### 5. Train (CoT lalu non-CoT)

In [ ]:
import gc, torch
for cfg_path in CONFIGS:
    cfg = load_config(cfg_path, max_examples=MAX_EXAMPLES)
    cfg.output_dir = f"{OUT_ROOT}/adapter_{Path(cfg_path).stem}"
    print('==='*12, cfg.mode, cfg.base_model, '->', cfg.output_dir)
    build_and_train(cfg)
    gc.collect(); torch.cuda.empty_cache()

### 6. Zip adapter -> Output
Eval pakai `notebooks/skenario4_eval_kaggle.ipynb` (load base + adapter sebagai 'model kita').

In [ ]:
!cd /kaggle/working && zip -rq adapters_sft.zip adapter_* && echo 'adapters_sft.zip siap di Output'